# torch-eml Quickstart

**EML tree heads for interpretable neuro-symbolic models.**

One function — `eml(x, y) = exp(x) - ln(y)` — recursed as a binary tree. Every elementary function emerges.

Train it like a neural network. Extract a closed-form equation. Publish it in your paper.

[![GitHub](https://img.shields.io/badge/GitHub-Gaius050526%2Ftorch--eml-blue)](https://github.com/Gaius050526/torch-eml)

In [ ]:
!pip install -q torch-eml

## 1. Discover sin(x) from data

In [ ]:
import torch
from torch_eml import EMLHead

torch.manual_seed(42)

# Generate noisy sin(x) data
X = torch.linspace(-3.14, 3.14, 500).unsqueeze(1)
y = torch.sin(X) + 0.02 * torch.randn_like(X)

# Train
head = EMLHead(n_inputs=1, depth=3)
opt = torch.optim.Adam(head.parameters(), lr=0.005)

for step in range(2000):
    loss = torch.nn.functional.mse_loss(head(X), y)
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f'Final loss: {loss.item():.6f}')

In [ ]:
# Prune dead branches, snap weights to clean values
head.prune(threshold=0.05, calibration_data=X)
expr = head.snap(tolerance=0.1, validation_data=(X, y))

print(f'Equation:  {expr.string}')
print(f'LaTeX:     {expr.latex}')
print(f'\nPython:\n{expr.python}')

## 2. Rediscover Kepler's Third Law

Given orbital radius `a`, discover that period `T = a^(3/2)` — from data alone.

In [ ]:
# Synthetic orbital data with noise
a = torch.rand(300, 1) * 5 + 0.5
T = (a ** 1.5) * (1 + 0.02 * torch.randn(300, 1))

head = EMLHead(n_inputs=1, depth=3)
opt = torch.optim.Adam(head.parameters(), lr=0.005)

for step in range(2000):
    loss = torch.nn.functional.mse_loss(head(a), T)
    opt.zero_grad()
    loss.backward()
    opt.step()

head.prune(threshold=0.1, calibration_data=a)
expr = head.snap(tolerance=0.15, validation_data=(a, T))
print(f'Discovered: T = {expr.string}')

## 3. Auto-tune depth + structure search

Don't know the right tree size? Let `search()` find it.

In [ ]:
from torch_eml import search

X = torch.linspace(-3.14, 3.14, 500).unsqueeze(1)
y = torch.sin(X)

result = search(n_inputs=1, X=X, y=y, max_depth=5, epochs=1000)

print(f'Best depth: {result.depth}')
print(f'Equation:   {result.expression.string}')
print(f'Loss:       {result.val_loss:.6f}')

## 4. Visualize the tree

In [ ]:
from torch_eml.viz import tree_to_html
from IPython.display import HTML

html = tree_to_html(result.head, title='Discovered Tree', equation=result.expression.string)
HTML(html)

## 5. MLP vs EML: same task, different outputs

Both fit `y = 3*sin(x1) + x2² - 0.5*x3`. Only one gives you the equation.

In [ ]:
import torch.nn as nn

# Dataset
X = torch.randn(500, 3) * 2
y = (3 * torch.sin(X[:, 0]) + X[:, 1]**2 - 0.5 * X[:, 2]).unsqueeze(1)

# MLP
mlp = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
opt = torch.optim.Adam(mlp.parameters(), lr=0.01)
for _ in range(2000):
    loss = nn.functional.mse_loss(mlp(X), y)
    opt.zero_grad(); loss.backward(); opt.step()
mlp_loss = loss.item()

# EML
head = EMLHead(n_inputs=3, depth=4)
opt = torch.optim.Adam(head.parameters(), lr=0.005)
for _ in range(3000):
    loss = nn.functional.mse_loss(head(X), y)
    opt.zero_grad(); loss.backward(); opt.step()

head.prune(threshold=0.05, calibration_data=X)
expr = head.snap(tolerance=0.1, validation_data=(X, y))
eml_loss = nn.functional.mse_loss(head(X), y).item()

print(f'MLP loss: {mlp_loss:.6f}  |  Equation: ???')
print(f'EML loss: {eml_loss:.6f}  |  Equation: {expr.string}')

---

**That's it.** Train like a neural net, extract a symbolic equation.

Full docs and more examples: [github.com/Gaius050526/torch-eml](https://github.com/Gaius050526/torch-eml)